## Natural Language Processing Spring 2026
---
# Tokenizer & Text Processing

# Introduction

**MiniMind Step-by-Step Series** | Milestone 1 of 6

In this first notebook, we build the foundation — a BPE tokenizer. By the end you'll understand how raw text becomes token IDs, how special tokens work, and how chat templates format conversations.

**Learning Objectives:**

- Train a Byte-Pair Encoding (BPE) tokenizer from scratch
- Understand vocabulary construction and special tokens
- Encode and decode text, verify lossless round-trips
- Apply chat templates for instruction-tuning format

In [1]:
# === Environment Setup ===
!pip install tokenizers torch transformers --quiet

import torch
import json
import math
print(f'PyTorch {torch.__version__}  CUDA available: {torch.cuda.is_available()}')

PyTorch 2.10.0+cu128  CUDA available: True


# Part A — Byte-Pair Encoding (BPE)

BPE iteratively merges the most frequent byte pairs into new tokens until the desired vocabulary size is reached. MiniMind uses a 6,400-token vocabulary — tiny compared to GPT-2's 50,257 but sufficient for a teaching model.

**Key concepts:**

- **ByteLevel pre-tokenizer** — operates on raw bytes so any UTF-8 string can be encoded
- **Special tokens** — reserved first in the vocabulary before training
- **After training** — the tokenizer can encode and decode any UTF-8 string

In [2]:
# === Train a BPE Tokenizer on a Small Corpus ===
from tokenizers import decoders, models, pre_tokenizers, trainers, Tokenizer

✅ Tokenizer trained — vocab size: 486
   Special tokens: 36


- tokenizers: This is the most crucial dependency in the notebook. It is a blazing-fast tokenizer building library written in Rust. The code imports several submodules from it to train a BPE (Byte-Pair Encoding) tokenizer from scratch:

    - Tokenizer: The main class used to instantiate and assemble the complete tokenizer.

    - models: Provides the underlying tokenization algorithm models. For example, models.BPE() used in this notebook stands for the Byte-Pair Encoding algorithm.

    - pre_tokenizers: Pre-tokenizers are responsible for the initial splitting of raw text before it is processed by the main model. ByteLevel is used here, treating text as a stream of raw bytes. This is a key technology ensuring the model can losslessly process any bizarre characters or multiple languages (such as Chinese, Emojis, etc.).

    - trainers: Provides utility classes for training the tokenizer on a corpus. BpeTrainer is used here to iterate through the corpus, count the frequencies of byte pairs, and generate the final vocabulary based on the specified vocabulary size (VOCAB_SIZE) and special tokens.

    - decoders: The decoder's function is the reverse of tokenization. It is responsible for decoding a sequence of Token IDs output by the model and joining them back into human-readable raw text.

In [5]:
VOCAB_SIZE = 6400
SPECIAL_TOKENS_NUM = 36

# Build a small demo corpus
corpus = [
    'The quick brown fox jumps over the lazy dog.',
    'MiniMind is a tiny language model for teaching.',
    'Byte-Pair Encoding merges frequent byte pairs.',
    'Attention is all you need.',
    'Transformers revolutionized natural language processing.',
    'Language models predict the next token in a sequence.',
    'Neural networks learn hierarchical representations.',
    'Training requires large amounts of text data.',
] * 100  # repeat so BPE has enough frequency signal

tokenizer = Tokenizer(models.BPE())
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)

# Define special tokens used in MiniMind chat format
all_special = [
    '<|endoftext|>', '<|im_start|>', '<|im_end|>', '<|im_sep|>',
    '<|endofprefix|>', '<|startoftext|>', '<|extra_0|>', '<|extra_1|>',
    '<|extra_2|>', '<|extra_3|>', '<|extra_4|>', '<|extra_5|>',
    '<|extra_6|>', '<|extra_7|>', '<|extra_8|>', '<|extra_9|>',
    '<pad>', '<mask>', '<sep>', '<cls>',
    '<unused_0>', '<unused_1>', '<unused_2>', '<unused_3>',
    '<unused_4>', '<unused_5>', '<unused_6>', '<unused_7>',
    '<think>', '</think>', '<tool_call>', '</tool_call>',
    '<tool_response>', '</tool_response>', '<code>', '</code>',
]

trainer_obj = trainers.BpeTrainer(
    vocab_size=VOCAB_SIZE,
    special_tokens=all_special,
    show_progress=True,
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
)

tokenizer.decoder = decoders.ByteLevel()
tokenizer.train_from_iterator(corpus, trainer=trainer_obj)

print(f' Tokenizer trained \u2014 vocab size: {tokenizer.get_vocab_size()}')
print(f'   Special tokens: {len(all_special)}')

✅ Tokenizer trained — vocab size: 486
   Special tokens: 36


## Inspecting the Vocabulary

The first tokens are special tokens, followed by byte-level tokens (256), then merged tokens learned by BPE.

In [9]:
# === Inspect the Vocabulary ===
vocab = tokenizer.get_vocab()
sorted_tokens = sorted(vocab.items(), key=lambda x: x[1])

print('First 20 tokens (includes special tokens):')
for tok, idx in sorted_tokens[:20]:
    print(f'  {idx:>5d}  {tok!r}')

print(f'\nByte-level tokens: 256')
print(f'Merged tokens: {tokenizer.get_vocab_size() - 256 - len(all_special)}')
print(f'Total vocabulary: {tokenizer.get_vocab_size()}')

First 20 tokens (includes special tokens):
      0  '<|endoftext|>'
      1  '<|im_start|>'
      2  '<|im_end|>'
      3  '<|im_sep|>'
      4  '<|endofprefix|>'
      5  '<|startoftext|>'
      6  '<|extra_0|>'
      7  '<|extra_1|>'
      8  '<|extra_2|>'
      9  '<|extra_3|>'
     10  '<|extra_4|>'
     11  '<|extra_5|>'
     12  '<|extra_6|>'
     13  '<|extra_7|>'
     14  '<|extra_8|>'
     15  '<|extra_9|>'
     16  '<pad>'
     17  '<mask>'
     18  '<sep>'
     19  '<cls>'

Byte-level tokens: 256
Merged tokens: 194
Total vocabulary: 486


## Encoding & Decoding

The tokenizer converts text to token IDs (encoding) and back (decoding). A good tokenizer should perform lossless round-trips — the decoded text must exactly match the original input.

In [4]:
# === Encode & Decode Demo ===
test_texts = [
    'Hello, MiniMind!',
    'The quick brown fox jumps.',
    'Attention is all you need.',
    '1 + 1 = 2',
]

for text in test_texts:
    ids = tokenizer.encode(text).ids
    decoded = tokenizer.decode(ids)
    print(f'Text:    {text!r}')
    print(f'Tokens:  {ids}')
    print(f'Decoded: {decoded!r}')
    print(f'Match:   {"\u2705" if decoded == text else "\u274c"}')
    print()

Text:    'Hello, MiniMind!'
Tokens:  [75, 104, 370, 114, 47, 256, 480, 36]
Decoded: 'Hello, MiniMind!'
Match:   ✅

Text:    'The quick brown fox jumps.'
Tokens:  [354, 462, 444, 450, 467, 49]
Decoded: 'The quick brown fox jumps.'
Match:   ✅

Text:    'Attention is all you need.'
Tokens:  [482, 340, 418, 404, 434, 49]
Decoded: 'Attention is all you need.'
Match:   ✅

Text:    '1 + 1 = 2'
Tokens:  [52, 256, 46, 256, 52, 256, 64, 256, 53]
Decoded: '1 + 1 = 2'
Match:   ✅



In [10]:
# === Milestone 1A: Lossless Encode/Decode Round-Trip ===
test_string = 'MiniMind is a tiny language model!'
encoded_ids = tokenizer.encode(test_string).ids
decoded_string = tokenizer.decode(encoded_ids)

assert decoded_string == test_string, (
    f'Round-trip failed!\n'
    f'  Original: {test_string!r}\n'
    f'  Decoded:  {decoded_string!r}'
)
print(f'\u2705 Milestone 1A passed \u2014 lossless encode/decode')
print(f'   "{test_string}" \u2192 {encoded_ids} \u2192 "{decoded_string}"')

✅ Milestone 1A passed — lossless encode/decode
   "MiniMind is a tiny language model!" → [480, 340, 300, 411, 343, 347, 36] → "MiniMind is a tiny language model!"


# Part B — Chat Templates

Modern LLMs use chat templates to format multi-turn conversations. MiniMind uses the ChatML format with `<|im_start|>` and `<|im_end|>` delimiters. This structured format tells the model which text is from the user vs the assistant.

In [11]:
# === Chat Template ===
def apply_chat_template(messages):
    """Apply MiniMind ChatML-style template."""
    text = ''
    for msg in messages:
        role = msg['role']
        content = msg['content']
        text += f'<|im_start|>{role}\n{content}<|im_end|>\n'
    return text

# Single-turn example
messages = [
    {'role': 'user', 'content': 'What is a language model?'},
    {'role': 'assistant', 'content': 'A language model predicts the next token in a sequence of text.'},
]
formatted = apply_chat_template(messages)
print('=== Formatted Chat ===')
print(formatted)

# Multi-turn example
multi_turn = [
    {'role': 'system', 'content': 'You are MiniMind, a helpful AI assistant.'},
    {'role': 'user', 'content': 'Hi!'},
    {'role': 'assistant', 'content': 'Hello! How can I help you?'},
    {'role': 'user', 'content': 'What is BPE?'},
    {'role': 'assistant', 'content': 'BPE stands for Byte-Pair Encoding, a tokenization algorithm.'},
]
formatted_multi = apply_chat_template(multi_turn)
print('=== Multi-Turn Chat ===')
print(formatted_multi)

# Encode the formatted text
encoded = tokenizer.encode(formatted)
print(f'Single-turn encoded ({len(encoded.ids)} tokens): {encoded.ids[:30]}...')

=== Formatted Chat ===
<|im_start|>user
What is a language model?<|im_end|>
<|im_start|>assistant
A language model predicts the next token in a sequence of text.<|im_end|>

=== Multi-Turn Chat ===
<|im_start|>system
You are MiniMind, a helpful AI assistant.<|im_end|>
<|im_start|>user
Hi!<|im_end|>
<|im_start|>assistant
Hello! How can I help you?<|im_end|>
<|im_start|>user
What is BPE?<|im_end|>
<|im_start|>assistant
BPE stands for Byte-Pair Encoding, a tokenization algorithm.<|im_end|>

Single-turn encoded (39 tokens): [1, 120, 118, 306, 234, 90, 107, 304, 340, 300, 343, 347, 66, 2, 234, 1, 100, 385, 328, 119, 100, 374, 234, 68, 343, 347, 476, 118, 342, 435]...


In [12]:
# === \u2705 Milestone 1B: Chat Template Verification ===
test_messages = [
    {'role': 'user', 'content': 'Hello'},
    {'role': 'assistant', 'content': 'Hi'},
]
result = apply_chat_template(test_messages)

assert '<|im_start|>user' in result, 'Missing user tag'
assert '<|im_start|>assistant' in result, 'Missing assistant tag'
assert '<|im_end|>' in result, 'Missing end tag'
assert result.count('<|im_start|>') == 2, f'Expected 2 start tags, got {result.count("<|im_start|>")}'
assert result.count('<|im_end|>') == 2, f'Expected 2 end tags, got {result.count("<|im_end|>")}'

print('\u2705 Milestone 1B passed \u2014 chat template works correctly')
print(f'   Template output:\n{result}')

✅ Milestone 1B passed — chat template works correctly
   Template output:
<|im_start|>user
Hello<|im_end|>
<|im_start|>assistant
Hi<|im_end|>



# Part C — Understanding Token IDs and Special Tokens

Special tokens like BOS (beginning of sequence) and EOS (end of sequence) are crucial signals. BOS tells the model "start generating" and EOS tells it "stop". The pad token fills sequences to uniform length for batching.

In [13]:
# === Special Token IDs ===
special_token_map = {
    'BOS (begin of sequence)': '<|im_start|>',
    'EOS (end of sequence)': '<|im_end|>',
    'PAD (padding)': '<pad>',
    'Think start': '<think>',
    'Think end': '</think>',
    'Tool call start': '<tool_call>',
    'Tool call end': '</tool_call>',
}

print('Special Token IDs:')
print('-' * 45)
vocab = tokenizer.get_vocab()
for desc, token in special_token_map.items():
    token_id = vocab.get(token, 'NOT FOUND')
    print(f'  {desc:25s} {token:20s} \u2192 {token_id}')

# Demonstrate padding to fixed length
MAX_LENGTH = 32
text = 'Hello, world!'
ids = tokenizer.encode(text).ids
pad_id = vocab.get('<pad>', 0)
padded = ids + [pad_id] * (MAX_LENGTH - len(ids))
print(f'\nOriginal tokens ({len(ids)}): {ids}')
print(f'Padded to {MAX_LENGTH}:       {padded}')

Special Token IDs:
---------------------------------------------
  BOS (begin of sequence)   <|im_start|>         → 1
  EOS (end of sequence)     <|im_end|>           → 2
  PAD (padding)             <pad>                → 16
  Think start               <think>              → 28
  Think end                 </think>             → 29
  Tool call start           <tool_call>          → 30
  Tool call end             </tool_call>         → 31

Original tokens (11): [75, 104, 370, 114, 47, 256, 122, 376, 111, 103, 36]
Padded to 32:       [75, 104, 370, 114, 47, 256, 122, 376, 111, 103, 36, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16]


In [15]:
# === Milestone 1C: Special Tokens & Padding ===
vocab = tokenizer.get_vocab()
assert '<|im_start|>' in vocab, 'Missing <|im_start|> special token'
assert '<|im_end|>' in vocab, 'Missing <|im_end|> special token'
assert '<pad>' in vocab, 'Missing <pad> special token'
assert '<think>' in vocab, 'Missing <think> special token'
assert '</think>' in vocab, 'Missing </think> special token'

# Verify padding
test_ids = tokenizer.encode('test').ids
pad_id = vocab['<pad>']
padded = test_ids + [pad_id] * (32 - len(test_ids))
assert len(padded) == 32, f'Padding failed: length {len(padded)} != 32'

print('Milestone 1C passed \u2014 all special tokens present, padding works')
print(f'   Found {len(all_special)} special tokens in vocabulary')

Milestone 1C passed — all special tokens present, padding works
   Found 36 special tokens in vocabulary


# Summary

| Component | Key Concept |
|---|---|
| BPE Tokenizer | Iterative byte-pair merging, 6400 tokens |
| Encode/Decode | Lossless text ↔ token ID conversion |
| Chat Template | ChatML format with im_start/im_end delimiters |
| Special Tokens | BOS, EOS, PAD, think, tool_call markers |

**Milestones completed:** 1A (round-trip), 1B (chat template), 1C (special tokens)

### Next Notebook

In **Notebook 2**, we'll build the model architecture: RMSNorm, Rotary Position Embeddings, Grouped-Query Attention, SwiGLU Feed-Forward, and assemble the complete MiniMind transformer.